# Nagpur Traffic Dash: Analysis of Play Data

This notebook looks at the data the game saves while people play.

- `data/runs.csv`: one line for every ride
- `data/crashes.csv`: one line for every crash

**If nobody has played yet, the files do not exist. The notebook then uses SIMULATED demo data.**
That data is marked "SIMULATED" everywhere, and it is never saved to disk.
Replace it by playing the game.

Run this notebook from the `Traffic_Dash` folder, so that the paths `data/...` work.

## 0. Setup
We import the libraries and our own `data.py` file (it cleans the obstacle table).
Run this cell first.

In [ ]:
import os
import numpy as np
import pandas as pd
import data

RUNS_PATH = os.path.join("data", "runs.csv")
CRASHES_PATH = os.path.join("data", "crashes.csv")

## 1. Load and inspect
We read both files. If they are missing, we make **SIMULATED** demo data instead.
The `if` checks whether each file exists, using `os.path.exists`.

In [ ]:
if os.path.exists(RUNS_PATH) and os.path.exists(CRASHES_PATH):
    runs_df = pd.read_csv(RUNS_PATH)
    crashes_df = pd.read_csv(CRASHES_PATH)
    print("Using the REAL data saved by playing the game.")
else:
    # SIMULATED - replace by playing the game
    print("*** SIMULATED DEMO DATA - NOT REAL PLAY. Replace by playing the game. ***")
    np.random.seed(42)
    n_runs = 30
    zone_names = ["Sitabuldi", "Dharampeth", "Wardha Road", "Ring Road"]
    zone_probs = [0.4, 0.3, 0.2, 0.1]
    obstacle_names = ["cow", "auto", "bus", "pothole", "cycle_rickshaw", "stray_dog", "barricade"]
    obstacle_probs = [0.25, 0.25, 0.1, 0.15, 0.1, 0.1, 0.05]

    # Random numbers for each simulated run
    distance = np.round(np.random.uniform(50, 1500, n_runs), 1)
    dodged = np.random.randint(0, 25, n_runs)
    powerups = np.random.randint(0, 6, n_runs)
    zone_reached = np.random.choice(zone_names, size=n_runs, p=zone_probs)

    # Each run gets 1 to 5 crashes, and each crash gets an obstacle and a second
    crash_rows = []
    crash_counts = []
    for i in range(n_runs):
        count = np.random.randint(1, 6)
        crash_counts.append(count)
        for k in range(count):
            crash_rows.append({"run_id": i + 1,
                               "obstacle": np.random.choice(obstacle_names, p=obstacle_probs),
                               "zone": zone_reached[i],
                               "second": np.random.randint(1, 120)})

    runs_df = pd.DataFrame({
        "run_id": np.arange(1, n_runs + 1),
        "player": "SIMULATED",
        "date": "SIMULATED",
        "distance_m": distance,
        "score": distance.astype(int) + dodged * 8,
        "time_s": np.round(distance / 13 + np.random.uniform(0, 5, n_runs), 1),
        "zone_reached": zone_reached,
        "dodged": dodged,
        "powerups": powerups,
        "crashes": crash_counts})
    crashes_df = pd.DataFrame(crash_rows)

print("runs_df shape:", runs_df.shape)
print("crashes_df shape:", crashes_df.shape)
print(runs_df.dtypes)
print(runs_df.head(4))
print(runs_df.tail(4))
print(crashes_df.dtypes)
print(crashes_df.head(4))
print(crashes_df.tail(4))

## 2. Cleaning demo
The raw obstacle file has mistakes on purpose: `??` and `##` typed instead of numbers,
a missing damage value, a name with spaces around it, and a repeated row.

Notice that `isnull()` on the raw file does **not** count `??` or `##`, because pandas reads them as text.
They only become missing values after we replace them. `data.load_obstacles()` does that cleaning.

In [ ]:
raw_obstacles = pd.read_csv(os.path.join("data", "obstacles_raw.csv"))
print("RAW file:")
print(raw_obstacles)
print("Missing values BEFORE cleaning:")
print(raw_obstacles.isnull().sum())
print("Data types BEFORE cleaning:")
print(raw_obstacles.dtypes)

clean_obstacles = data.load_obstacles()
print("Missing values AFTER cleaning:")
print(clean_obstacles.isnull().sum())
print("Data types AFTER cleaning:")
print(clean_obstacles.dtypes)
print("CLEAN table:")
print(clean_obstacles)

## 3. Descriptive statistics
`describe()` gives the summary. Then we print the mean, median, mode, variance and skew for each column.
A skew above 0 means a long tail of big values; below 0 means a long tail of small values.

In [ ]:
stats_cols = ["score", "distance_m", "time_s"]
print(runs_df[stats_cols].describe())
for col in stats_cols:
    column = runs_df[col]
    print("{:12s} mean={:.2f} median={:.2f} mode={} var={:.2f} skew={:.2f}".format(
        col, column.mean(), column.median(), column.mode()[0], column.var(), column.skew()))

## 4. Group analysis
`groupby` puts the rows into groups (for example, one group per obstacle) and counts or averages each group.

In [ ]:
print("Crashes per obstacle:")
print(crashes_df.groupby("obstacle").size().sort_values(ascending=False))

print("Crashes per zone:")
print(crashes_df.groupby("zone").size().sort_values(ascending=False))

print("Average score and time for each zone reached:")
print(runs_df.groupby("zone_reached")[["score", "time_s"]].mean())

## 5. Extremes and leaderboard
`idxmax()` and `idxmin()` give the row position of the biggest and smallest score.
`sort_values` gives the leaderboard.

In [ ]:
best_index = runs_df["score"].idxmax()
worst_index = runs_df["score"].idxmin()
print("Best run:")
print(runs_df.loc[best_index])
print("Worst run:")
print(runs_df.loc[worst_index])

leaderboard = runs_df.sort_values("score", ascending=False)
print("Leaderboard (top 5):")
print(leaderboard[["player", "score", "zone_reached"]].head(5))

## 6. Correlation
`corr()` shows how strongly two columns move together. A value near 1 or -1 is a strong link; near 0 is no link.
Correlation does not prove that one column causes the other.

In [ ]:
print(runs_df[["score", "time_s", "dodged", "powerups", "crashes"]].corr())

## 7. Pearson r by hand
We compare `powerups` and `score` using the formula

r = (n·Σxy − Σx·Σy) / sqrt((n·Σx² − (Σx)²) · (n·Σy² − (Σy)²))

We compute it with NumPy and then check it against `.corr()`. The two numbers should match.

In [ ]:
x = np.array(runs_df["powerups"], dtype=float)
y = np.array(runs_df["score"], dtype=float)
n = len(x)

top_part = n * np.sum(x * y) - np.sum(x) * np.sum(y)
left_bottom = n * np.sum(x ** 2) - np.sum(x) ** 2
right_bottom = n * np.sum(y ** 2) - np.sum(y) ** 2
r_by_hand = top_part / np.sqrt(left_bottom * right_bottom)

print("Pearson r by hand: {:.4f}".format(r_by_hand))
print("Pearson r with .corr(): {:.4f}".format(runs_df["powerups"].corr(runs_df["score"])))

## 8. New column: points per second
`points_per_sec` is the score divided by the time. We show the top 5 runs by this value.

In [ ]:
runs_df["points_per_sec"] = runs_df["score"] / runs_df["time_s"]
top5_rate = runs_df.sort_values("points_per_sec", ascending=False).head(5)
print(top5_rate[["run_id", "score", "time_s", "points_per_sec"]])

## 9. Improvement
We compare the average score of the first 5 runs with the average of the latest 5 runs.
A positive number means the player got better. We need at least 10 runs for this to mean something.

In [ ]:
scores = np.array(runs_df["score"])
if len(scores) < 10:
    print("Only {} runs so far. Play at least 10 runs for a fair comparison.".format(len(scores)))

first5 = scores[:5].mean()
latest5 = scores[-5:].mean()
print("Average of the first 5 runs: {:.1f}".format(first5))
print("Average of the latest 5 runs: {:.1f}".format(latest5))
print("Improvement: {:+.1f}".format(latest5 - first5))

## 10. Findings
Fill in these sentences with what the data shows. Use numbers from the cells above.

1. The obstacle that causes the most crashes is **____**, with **____** crashes. I think this is because ____.
2. The zone where players score the most on average is **____**. I think this is because ____.
3. The correlation between powerups and score is **____** (r = ____). This means ____.
4. The biggest surprise in the data is ____.